In [ ]:
# =========================================
# Road Safety Data Analysis Script
# =========================================

import pandas as pd
import numpy as np

# -----------------------------------------
# 1. Load datasets
# -----------------------------------------

casualties = pd.read_csv("dft-road-casualty-statistics-casualty-1979-latest-published-year.csv")
collisions = pd.read_csv("dft-road-casualty-statistics-collision-1979-latest-published-year.csv")
vehicles = pd.read_csv("dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv")


# -----------------------------------------
# 2. Select relevant columns and filter
#    to 2015 onwards
# -----------------------------------------

casualties_1 = casualties[
    [
        "collision_ref_no",
        "collision_year",
        "vehicle_reference",
        "casualty_reference",
        "sex_of_casualty",
        "age_of_casualty",
        "casualty_severity",
        "casualty_type",
    ]
]

casualties_1 = casualties_1[casualties_1["collision_year"] >= 2015]


collisions_1 = collisions[
    [
        "collision_ref_no",
        "collision_year",
        "longitude",
        "latitude",
        "date",
        "day_of_week",
        "time",
        "local_authority_district",
        "local_authority_ons_district",
        "first_road_class",
        "road_type",
        "speed_limit",
        "junction_detail",
        "junction_control",
        "pedestrian_crossing",
        "light_conditions",
        "urban_or_rural_area",
    ]
]

collisions_1 = collisions_1[collisions_1["collision_year"] >= 2015]


vehicles_1 = vehicles[
    [
        "collision_ref_no",
        "collision_year",
        "vehicle_reference",
        "vehicle_type",
    ]
]

vehicles_1 = vehicles_1[vehicles_1["collision_year"] >= 2015]


# -----------------------------------------
# 3. Merge datasets
# -----------------------------------------

# Merge casualties with collisions
cas_coll = pd.merge(
    casualties_1,
    collisions_1,
    on=["collision_ref_no", "collision_year"],
    how="left"
)

# Merge vehicles
full_data = pd.merge(
    cas_coll,
    vehicles_1,
    on=["collision_ref_no", "collision_year", "vehicle_reference"],
    how="left"
)


# -----------------------------------------
# 4. Filter for KSI (Killed or Seriously Injured)
# -----------------------------------------

ksi_data = full_data[full_data["casualty_severity"].isin([1, 2])]


# -----------------------------------------
# 5. Define key variables needed for analysis
# -----------------------------------------

key_cols = [
    "collision_year",
    "longitude",
    "latitude",
    "date",
    "day_of_week",
    "time",
    "local_authority_district",
    "first_road_class",
    "road_type",
    "speed_limit",
    "junction_detail",
    "junction_control",
    "pedestrian_crossing",
    "light_conditions",
    "urban_or_rural_area",
    "vehicle_type",
]


# -----------------------------------------
# 6. Keep only rows with complete key variables
# -----------------------------------------

ksi_data_code_match = ksi_data.dropna(subset=key_cols)


# -----------------------------------------
# 7. Replace "-1" coded missing values with NA
# -----------------------------------------

ksi_data = ksi_data.replace(-1, np.nan)
ksi_data_code_match = ksi_data_code_match.replace(-1, np.nan)


# -----------------------------------------
# 8. Missing value summary (Full KSI dataset)
# -----------------------------------------

na_summary = ksi_data.isna().sum().reset_index()
na_summary.columns = ["column", "na_count"]

total_rows = len(ksi_data)

na_summary["total_rows"] = total_rows
na_summary["na_percent"] = (na_summary["na_count"] / total_rows) * 100

print("\nMissing Value Summary (Full KSI Dataset):")
print(na_summary)


# -----------------------------------------
# 9. Missing value summary (Clean dataset)
# -----------------------------------------

na_summary_code_match = ksi_data_code_match.isna().sum().reset_index()
na_summary_code_match.columns = ["column", "na_count"]

total_rows_clean = len(ksi_data_code_match)

na_summary_code_match["total_rows"] = total_rows_clean
na_summary_code_match["na_percent"] = (
    na_summary_code_match["na_count"] / total_rows_clean
) * 100

print("\nMissing Value Summary (Cleaned Dataset):")
print(na_summary_code_match)